In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, StratifiedKFold, GridSearchCV,
    learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")

### Task 1

In [ ]:
digits = load_digits()

# Convert to DataFrame
df = pd.DataFrame(digits.data, columns=[f'pixel_{i}' for i in range(digits.data.shape[1])])
df['target'] = digits.target

print(f"Dataset shape      : {df.shape}")
print(f"Number of classes  : {len(np.unique(digits.target))}")
print(f"Features per sample: {digits.data.shape[1]}")

# Class distribution
class_counts = df['target'].value_counts().sort_index()
print("\nClass distribution:")
print(class_counts.to_string())
print(f"\nMin count: {class_counts.min()} | Max count: {class_counts.max()}")
print(f"Imbalance ratio (max/min): {class_counts.max()/class_counts.min():.2f}")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="Blues_d", ax=ax)
ax.set_xlabel("Digit Class")
ax.set_ylabel("Count")
ax.set_title("Class Distribution — Digits Dataset")
for bar, val in zip(ax.patches, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LogisticRegression(max_iter=5000, random_state=42)
lr.fit(X_train, y_train)
single_split_acc = accuracy_score(y_test, lr.predict(X_test))

print(f"Training samples  : {X_train.shape[0]}")
print(f"Test samples      : {X_test.shape[0]}")
print(f"Hold-out Accuracy : {single_split_acc:.4f}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_kf = LogisticRegression(max_iter=5000, random_state=42)

kfold_scores = cross_val_score(lr_kf, X, y, cv=kf, scoring='accuracy')

print(f"KFold scores : {np.round(kfold_scores, 4)}")
print(f"Mean         : {kfold_scores.mean():.4f}")
print(f"Std          : {kfold_scores.std():.4f}")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_skf = LogisticRegression(max_iter=5000, random_state=42)

skfold_scores = cross_val_score(lr_skf, X, y, cv=skf, scoring='accuracy')

print(f"Stratified KFold scores : {np.round(skfold_scores, 4)}")
print(f"Mean                    : {skfold_scores.mean():.4f}")
print(f"Std                     : {skfold_scores.std():.4f}")

In [ ]:
methods = ['Hold-out\n(Single Split)', '5-Fold CV\n(KFold)', '5-Fold CV\n(Stratified)']
means   = [single_split_acc, kfold_scores.mean(), skfold_scores.mean()]
errors  = [0, kfold_scores.std(), skfold_scores.std()]
colors  = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, means, yerr=errors, capsize=8, color=colors,
              error_kw=dict(elinewidth=2, ecolor='black'), width=0.5)

ax.set_ylim(0.92, 1.00)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy Comparison Across Splitting Strategies\n(LogisticRegression, Digits Dataset)")

for bar, mean, err in zip(bars, means, errors):
    label = f"{mean:.4f}" + (f"\n±{err:.4f}" if err > 0 else "")
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


Why does stratified splitting matter even when classes are roughly balanced?

Even with near-equal class counts, random splits can still produce folds where one class is over- or under-represented by chance — especially with smaller datasets. Stratification guarantees that each fold mirrors the full dataset's class proportions, making each validation score a fair and consistent estimate of generalization performance.

When would you expect a larger gap between regular and stratified CV?

The difference grows significantly when: (1) the dataset is small, so random variation in fold composition is larger; (2) the class distribution is highly imbalanced (e.g., 95/5 split), where a random fold could contain almost no minority-class samples; or (3) the model is sensitive to class representation — like SVC or KNN — where seeing too few examples of a class in training directly hurts predictions on that class.

### Task 2

In [ ]:
from sklearn.metrics import make_scorer

# Define the 5 classifiers
classifiers = {
    'LogisticRegression'       : LogisticRegression(max_iter=5000, random_state=42),
    'SVC'                      : SVC(random_state=42),
    'RandomForest'             : RandomForestClassifier(random_state=42),
    'GradientBoosting'         : GradientBoostingClassifier(random_state=42),
    'KNeighbors'               : KNeighborsClassifier()
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy' : 'accuracy',
    'precision': 'precision_macro',
    'recall'   : 'recall_macro',
    'f1'       : 'f1_macro'
}

# Run CV for each model and collect results
from sklearn.model_selection import cross_validate

results_raw = {}
for name, model in classifiers.items():
    print(f"Evaluating {name}...")
    cv_results = cross_validate(model, X, y, cv=skf, scoring=scoring)
    results_raw[name] = cv_results
    
print("\nDone!")

In [ ]:
rows = []
for name, res in results_raw.items():
    acc  = res['test_accuracy']
    prec = res['test_precision']
    rec  = res['test_recall']
    f1   = res['test_f1']
    rows.append({
        'Model'              : name,
        'Accuracy (mean±std)': f"{acc.mean():.4f} ± {acc.std():.4f}",
        'Precision (macro)'  : f"{prec.mean():.4f} ± {prec.std():.4f}",
        'Recall (macro)'     : f"{rec.mean():.4f} ± {rec.std():.4f}",
        'F1 (macro)'         : f"{f1.mean():.4f} ± {f1.std():.4f}",
        '_f1_mean'           : f1.mean()   # helper column for sorting
    })

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values('_f1_mean', ascending=False).reset_index(drop=True)
comparison_df = comparison_df.drop(columns='_f1_mean')

print(comparison_df.to_string(index=False))

In [ ]:
# Extract numeric means/stds for plotting
plot_data = []
for name, res in results_raw.items():
    plot_data.append({
        'Model': name,
        'f1_mean': res['test_f1'].mean(),
        'f1_std' : res['test_f1'].std(),
        'acc_mean': res['test_accuracy'].mean(),
        'acc_std' : res['test_accuracy'].std(),
    })

plot_df = pd.DataFrame(plot_data).sort_values('f1_mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# F1 Score
axes[0].barh(plot_df['Model'], plot_df['f1_mean'],
             xerr=plot_df['f1_std'], capsize=5,
             color='#4C72B0', edgecolor='white')
axes[0].set_xlabel("F1 Score (macro)")
axes[0].set_title("F1 Score by Model (Stratified 5-Fold CV)")
axes[0].set_xlim(0.85, 1.01)
for i, (mean, std) in enumerate(zip(plot_df['f1_mean'], plot_df['f1_std'])):
    axes[0].text(mean + 0.001, i, f"{mean:.4f}", va='center', fontsize=9)

# Accuracy
axes[1].barh(plot_df['Model'], plot_df['acc_mean'],
             xerr=plot_df['acc_std'], capsize=5,
             color='#55A868', edgecolor='white')
axes[1].set_xlabel("Accuracy")
axes[1].set_title("Accuracy by Model (Stratified 5-Fold CV)")
axes[1].set_xlim(0.85, 1.01)
for i, (mean, std) in enumerate(zip(plot_df['acc_mean'], plot_df['acc_std'])):
    axes[1].text(mean + 0.001, i, f"{mean:.4f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

Top 2 models: SVC (F1: 0.9877) and KNeighbors (F1: 0.9872)

The ranking is somewhat surprising — KNeighbors beating RandomForest and LogisticRegression is notable. KNN works exceptionally well here because digit images that look visually similar (close in pixel space) genuinely belong to the same class, so proximity in 64-dimensional space is a meaningful signal.

SVC takes the top spot as expected: the RBF kernel maps pixel features into a higher-dimensional space where digit classes become linearly separable, and the margin maximization makes it robust against noise in individual pixels.

GradientBoosting finishing last is not surprising — it's designed for structured/tabular data and builds trees sequentially without the spatial awareness that benefits SVC and KNN on image-like data.

Smallest variance: SVC has the tightest std (±0.0042) vs KNN's (±0.0080), meaning SVC is both more accurate on average and more consistent fold-to-fold. It's the safer production choice.


### Task 3

In [ ]:
# Hyperparameter grids for the top 2 models
param_grid_svc = {
    'C'      : [0.1, 1, 10, 100],
    'kernel' : ['rbf', 'poly'],
    'gamma'  : ['scale', 'auto']
}

param_grid_knn = {
    'n_neighbors' : [3, 5, 7, 9, 11],
    'weights'     : ['uniform', 'distance'],
    'metric'      : ['euclidean', 'manhattan', 'minkowski']
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV for SVC
print("Running GridSearchCV for SVC...")
grid_svc = GridSearchCV(
    SVC(random_state=42), param_grid_svc,
    cv=skf, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_svc.fit(X_train, y_train)

# GridSearchCV for KNeighbors
print("\nRunning GridSearchCV for KNeighbors...")
grid_knn = GridSearchCV(
    KNeighborsClassifier(), param_grid_knn,
    cv=skf, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_knn.fit(X_train, y_train)

print("\nDone!")

In [ ]:
print("=" * 50)
print("SVC — Best Parameters:")
for k, v in grid_svc.best_params_.items():
    print(f"  {k}: {v}")
print(f"  Best CV F1 (macro): {grid_svc.best_score_:.4f}")

print()
print("=" * 50)
print("KNeighbors — Best Parameters:")
for k, v in grid_knn.best_params_.items():
    print(f"  {k}: {v}")
print(f"  Best CV F1 (macro): {grid_knn.best_score_:.4f}")

In [ ]:
# Default F1 scores from Task 2
default_svc_f1 = 0.9877
default_knn_f1 = 0.9872

tuned_svc_f1 = grid_svc.best_score_
tuned_knn_f1 = grid_knn.best_score_

# Summary table
tuning_summary = pd.DataFrame({
    'Model'    : ['SVC', 'KNeighbors'],
    'Default F1 (CV)': [default_svc_f1, default_knn_f1],
    'Tuned F1 (CV)'  : [tuned_svc_f1,   tuned_knn_f1],
    'Delta'          : [tuned_svc_f1 - default_svc_f1,
                        tuned_knn_f1 - default_knn_f1]
})
tuning_summary['Delta'] = tuning_summary['Delta'].map(lambda x: f"+{x:.4f}" if x >= 0 else f"{x:.4f}")
print(tuning_summary.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)
width = 0.35
bars1 = ax.bar(x - width/2, [default_svc_f1, default_knn_f1],
               width, label='Default', color='#4C72B0')
bars2 = ax.bar(x + width/2, [tuned_svc_f1, tuned_knn_f1],
               width, label='Tuned', color='#DD8452')

ax.set_xticks(x)
ax.set_xticklabels(['SVC', 'KNeighbors'])
ax.set_ylim(0.97, 1.00)
ax.set_ylabel("F1 Score (macro)")
ax.set_title("Default vs Tuned F1 Score (GridSearchCV)")
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f"{bar.get_height():.4f}", ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f"{bar.get_height():.4f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

Results: SVC improved, KNeighbors slightly regressed
SVC gained a meaningful +0.0025 improvement through tuning — confirming that the default C=1 was leaving some margin on the table. A higher C value reduces regularization and allows the model to fit the digit boundaries more tightly, which pays off here since the training data is clean and low-noise.

KNeighbors actually dropped by -0.0011, which is a useful reminder that GridSearchCV can occasionally overfit the validation folds — the "best" parameter combination on CV splits may not generalize better than the default. The default k=5, weights='uniform' was already well-suited for this dataset's geometry.

Best overall tuned model: SVC with a CV F1 of 0.9902, which will carry forward into Task 4.

Was tuning worth the cost?

On this dataset size, absolutely — it runs in seconds. At 100× scale (~180k samples), kernel SVM becomes impractical (it scales O(n²)–O(n³)), so the strategy would shift entirely: use LinearSVC or SGDClassifier, switch from GridSearchCV to RandomizedSearchCV to sample the parameter space, reduce to 3-fold CV, and consider HalvingGridSearchCV which eliminates weak candidates early and cuts compute significantly.

### Task 4

In [ ]:
# Best model is the tuned SVC from Task 3
best_model = grid_svc.best_estimator_

# Train on full training set, predict on test set
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Best SVC parameters:", grid_svc.best_params_)
print(f"\nTest Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Test F1 (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=digits.target_names)
disp.plot(ax=ax, cmap='Blues', colorbar=True)

ax.set_title("Confusion Matrix — Tuned SVC (Digits Dataset)", fontsize=13, pad=15)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)

plt.tight_layout()
plt.show()

# Print the worst-confused pairs
print("Most confused digit pairs (off-diagonal errors):")
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
for _ in range(5):
    idx = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
    if cm_no_diag[idx] > 0:
        print(f"  True={idx[0]} → Predicted={idx[1]}  ({cm_no_diag[idx]} times)")
        cm_no_diag[idx] = 0

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    grid_svc.best_estimator_, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='f1_macro',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(train_sizes, train_mean, 'o-', color='#4C72B0', label='Training Score')
ax.fill_between(train_sizes, train_mean - train_std,
                train_mean + train_std, alpha=0.15, color='#4C72B0')

ax.plot(train_sizes, val_mean, 'o-', color='#DD8452', label='Validation Score')
ax.fill_between(train_sizes, val_mean - val_std,
                val_mean + val_std, alpha=0.15, color='#DD8452')

ax.set_xlabel("Training Set Size", fontsize=12)
ax.set_ylabel("F1 Score (macro)", fontsize=12)
ax.set_title("Learning Curves — Tuned SVC", fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim(0.85, 1.01)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
perm_imp = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=30, random_state=42, scoring='f1_macro', n_jobs=-1
)

# Sort and take top 20
sorted_idx = perm_imp.importances_mean.argsort()[::-1][:20]
top_features     = [f'pixel_{i}' for i in sorted_idx]
top_means        = perm_imp.importances_mean[sorted_idx]
top_stds         = perm_imp.importances_std[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_features[::-1], top_means[::-1],
        xerr=top_stds[::-1], capsize=4,
        color='#4C72B0', edgecolor='white')
ax.set_xlabel("Mean Decrease in F1 (macro)", fontsize=11)
ax.set_title("Top 20 Permutation Importances — Tuned SVC", fontsize=13)
ax.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Also visualise importances mapped onto the 8x8 pixel grid
imp_grid = perm_imp.importances_mean.reshape(8, 8)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(imp_grid, cmap='YlOrRd', ax=ax, cbar=True,
            linewidths=0.5, annot=False)
ax.set_title("Permutation Importance — 8×8 Pixel Grid", fontsize=12)
ax.set_xlabel("Pixel Column")
ax.set_ylabel("Pixel Row")
plt.tight_layout()
plt.show()

Overfitting, underfitting, or well-fitted?

The model is well-fitted. The learning curves show training and validation scores converging closely as data increases, with no sign of divergence (overfitting) or a low plateau (underfitting).

Most confused digit pairs?

Errors are very rare (1 occurrence each) and spread across pairs: 5→6, 7→9, 8→5, 9→3, 9→7. These all involve digits that share similar curved or angular strokes in the 8×8 pixel space — 5/6 and 7/9 are the classic lookalikes in handwriting.

Which pixel positions matter most?

The most important pixels cluster in the center of the 8×8 grid (rows 2–6, columns 2–6) where ink strokes actually appear. Corner and edge pixels are near-zero — they're almost always blank and carry no discriminating information.

What would you try next?

Remove the near-zero-importance corner pixels via feature selection to speed up inference with no accuracy cost.

Experiment with a linear kernel or LinearSVC now that we know C=0.1 (strong regularization) already performs well — it may be faster with similar accuracy.

Try soft probability outputs via CalibratedClassifierCV to get confidence scores alongside predictions, useful for flagging ambiguous samples like the 9→3 and 9→7 confusions.